# Stage 4: Modelling design and data partitioning

Stage 4 defines the modelling design before model fitting. It creates the held-out test partition, nested cross-validation folds, predictor preprocessing roles, model-search rules, evaluation measures and predictor-domain contribution design.

No model is fitted in this stage.

## Part 1: Design principles

1. The held-out test sample is reserved for final evaluation.
2. Hyperparameter selection is performed in inner cross-validation folds; outer folds estimate development performance.
3. Macro F1 is the single hyperparameter-selection criterion.
4. Overall and class-specific metrics are reported for evaluation.
5. Imbalance handling is fitted within training data only.

In [1]:
# 1: Project paths and imports

from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split, StratifiedKFold

working_directory = Path.cwd().resolve()

# Locate the project root when the notebook is run from the root or a subdirectory.
project_root = next(
    (
        directory
        for directory in [working_directory, *working_directory.parents]
        if (
            directory
            / "data_derived"
            / "stage_3_final_modelling_dataset"
            / "final_modelling_dataset.csv"
        ).is_file()
        and (
            directory
            / "data_derived"
            / "stage_2_predictor_construction"
            / "stage_2_final_predictor_manifest.csv"
        ).is_file()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "The Stage 2 predictor manifest and Stage 3 modelling dataset could not be located."
    )

data_derived = project_root / "data_derived"

stage_2_directory = data_derived / "stage_2_predictor_construction"
stage_3_directory = data_derived / "stage_3_final_modelling_dataset"
stage_4_directory = data_derived / "stage_4_modelling_design"
stage_4_directory.mkdir(parents=True, exist_ok=True)

modelling_data_path = stage_3_directory / "final_modelling_dataset.csv"
predictor_manifest_path = stage_2_directory / "stage_2_final_predictor_manifest.csv"

print("Project root located.")
print(
    "Stage 4 output directory:",
    stage_4_directory.relative_to(project_root).as_posix(),
)

Project root located.
Stage 4 output directory: data_derived/stage_4_modelling_design


In [2]:
# 2: Load modelling data and predictor manifest

# Load the analytic sample and the registered predictor definitions.
modelling_data = pd.read_csv(modelling_data_path, dtype={"NSID": "string"})
predictor_manifest = pd.read_csv(predictor_manifest_path)

predictor_columns = [
    column for column in modelling_data.columns
    if column not in {"NSID", "age18_outcome_code", "age18_outcome"}
]

print(f"Modelling data shape: {modelling_data.shape}")
print(f"Predictors: {len(predictor_columns)}")
print(
    "Represented predictor domains:",
    predictor_manifest["Predictor domain"].nunique(),
)


Modelling data shape: (9524, 72)
Predictors: 69
Represented predictor domains: 10


In [3]:
# 3: Input integrity checks

# Confirm the modelling sample, outcome and predictor register before partitioning.

expected_outcome_codes = {1, 4, 5, 6}

assert modelling_data.shape == (9_524, 72)
assert modelling_data["NSID"].notna().all()
assert modelling_data["NSID"].is_unique
assert modelling_data[["age18_outcome_code", "age18_outcome"]].notna().all().all()
assert set(modelling_data["age18_outcome_code"]) == expected_outcome_codes
assert len(predictor_columns) == 69
assert predictor_manifest["Predictor"].is_unique
assert set(predictor_manifest["Predictor"]) == set(predictor_columns)

outcome_mapping = (
    modelling_data[["age18_outcome_code", "age18_outcome"]]
    .drop_duplicates()
    .sort_values("age18_outcome_code")
    .reset_index(drop=True)
)

print("Outcome mapping:")
display(outcome_mapping)

print("Outcome counts:")
display(
    modelling_data["age18_outcome"]
    .value_counts()
    .rename_axis("Outcome")
    .reset_index(name="Participants")
)


Outcome mapping:


,age18_outcome_code,age18_outcome
0,1,Education
1,4,Employment
2,5,Apprenticeship or training
3,6,Unemployment or inactivity (NEET)


Outcome counts:


,Outcome,Participants
0,Education,4952
1,Employment,2801
2,Unemployment or inactivity (NEET),1251
3,Apprenticeship or training,520


## Part 2: Predictor representation

- Numeric and ordinal predictors: median imputation.
- Binary and nominal predictors: most-frequent imputation and one-hot encoding.
- Numeric and ordinal predictors are standardised for multinomial logistic regression only.
- Tree-based models are not scaled.

Routed profiles without a clear substantive order are treated as nominal.

In [4]:
# 4: Numeric and ordinal predictors

# Preserve continuous values and substantive ordering without one-hot expansion.

numeric_predictors = [
    "birth_month_position",
    "working_parent_or_guardian_count",
    "school_attitude_score",
    "homework_evenings_pretransition",
    "teacher_listening_fair_treatment_score_pretransition",
    "academic_self_concept_score",
    "young_person_ghq12_score",
    "parent_relationship_quality_score_pretransition",
    "parent_communication_frequency_score_pretransition",
    "parental_autonomy_support_score_pretransition",
    "teacher_guidance_frequency_index_pretransition",
]

ordinal_predictors = [
    "highest_parental_qualification_code",
    "household_income_band",
    "sen_status_pretransition",
    "disability_schooling_category",
    "main_parent_disability",
    "young_person_general_health",
    "main_parent_general_health",
    "higher_education_application_likelihood",
    "school_physical_environment_perception_pretransition",
    "smoking_activity_pretransition",
    "alcohol_use_frequency_pretransition",
    "recent_antisocial_behaviour_pretransition",
    "parental_higher_education_expectation_pretransition",
    "parental_educational_aspiration_pretransition",
    "parental_educational_financial_support_profile_pretransition",
    "homework_monitoring_frequency_pretransition",
    "parental_knowledge_of_evening_whereabouts_pretransition",
    "school_night_curfew_pretransition",
    "parent_school_day_discussion_frequency_pretransition",
    "shared_family_activity_frequency_pretransition",
    "family_future_study_discussion_frequency_pretransition",
    "peer_future_study_discussion_frequency_pretransition",
    "parental_training_apprenticeship_discussion_profile_pretransition",
    "careers_advice_service_guidance_frequency_pretransition",
    "learning_mentor_engagement_profile_pretransition",
    "stay_on_guidance_profile_pretransition",
    "apprenticeship_guidance_profile_pretransition",
]


In [5]:
# 5: Binary and nominal predictors

# Treat unordered categories as categorical inputs for one-hot encoding.

binary_predictors = [
    "sex",
    "english_first_or_main_language",
    "non_english_home_language",
    "home_computer_access_pretransition",
    "current_statement_of_needs_pretransition",
    "additional_parental_care_due_to_disability",
    "disability_benefit_receipt",
    "truancy_status_pretransition",
    "any_school_change_by_wave3",
    "school_exclusion_history_pretransition",
    "vocational_course_study_pretransition",
    "bullying_experience_pretransition",
    "young_person_caring_responsibility_pretransition",
    "cannabis_experience_pretransition",
    "recent_police_contact_pretransition",
    "homework_help_at_home_pretransition",
    "parents_evening_attendance_pretransition",
    "specially_arranged_teacher_meeting_ever_pretransition",
    "school_contact_about_behaviour_ever_pretransition",
    "connexions_adviser_contact_pretransition",
    "independent_school_at_sampling_stage",
]

nominal_predictors = [
    "ethnicity",
    "family_nssec_code",
    "housing_tenure",
    "family_composition",
    "expected_post16_route",
    "cumulative_school_history_pretransition",
    "parent_teacher_post16_discussion_profile_pretransition",
    "expected_peer_post16_route",
    "urban_rural_settlement_type_pretransition",
    "government_office_region_pretransition",
]


In [6]:
# 6: Predictor-role register and checks

# Map each predictor to one preprocessing route and save the register for Stage 5.

role_lists = {
    "Numeric": numeric_predictors,
    "Ordinal": ordinal_predictors,
    "Binary": binary_predictors,
    "Nominal": nominal_predictors,
}

role_records = []
for measurement_role, predictors in role_lists.items():
    for predictor in predictors:
        role_records.append({
            "Predictor": predictor,
            "Measurement role": measurement_role,
        })

predictor_roles = pd.DataFrame(role_records)
predictor_roles = predictor_manifest[["Predictor", "Predictor domain"]].merge(
    predictor_roles,
    on="Predictor",
    how="left",
    validate="one_to_one",
)

predictor_roles["Preprocessing role"] = np.where(
    predictor_roles["Measurement role"].isin(["Numeric", "Ordinal"]),
    "Numeric",
    "Categorical",
)
predictor_roles["Imputation"] = np.where(
    predictor_roles["Preprocessing role"].eq("Numeric"),
    "Median",
    "Most frequent",
)
predictor_roles["Encoding"] = np.where(
    predictor_roles["Preprocessing role"].eq("Categorical"),
    "One-hot; drop if binary",
    "None",
)
predictor_roles["MLR scaling"] = np.where(
    predictor_roles["Preprocessing role"].eq("Numeric"),
    "Standard scaling",
    "None",
)

assert len(predictor_roles) == 69
assert predictor_roles["Predictor"].is_unique
assert predictor_roles["Measurement role"].notna().all()
assert set(predictor_roles["Predictor"]) == set(predictor_columns)
assert predictor_roles["Measurement role"].value_counts().to_dict() == {
    "Ordinal": 27,
    "Binary": 21,
    "Numeric": 11,
    "Nominal": 10,
}

for predictor in numeric_predictors + ordinal_predictors:
    converted = pd.to_numeric(modelling_data[predictor], errors="coerce")
    assert converted.notna().sum() == modelling_data[predictor].notna().sum()

for predictor in binary_predictors:
    assert modelling_data[predictor].dropna().nunique() <= 2

print("Measurement roles:")
display(
    predictor_roles["Measurement role"]
    .value_counts()
    .rename_axis("Role")
    .reset_index(name="Predictors")
)

predictor_roles_path = stage_4_directory / "stage_4_predictor_preprocessing_roles.csv"
predictor_roles.to_csv(predictor_roles_path, index=False)
print(f"Saved: {predictor_roles_path.relative_to(project_root)}")


Measurement roles:


,Role,Predictors
0,Ordinal,27
1,Binary,21
2,Numeric,11
3,Nominal,10


Saved: data_derived\stage_4_modelling_design\stage_4_predictor_preprocessing_roles.csv


## Part 3: Held-out test partition

The participant-level stratified 80:20 split creates the training sample for model development and the held-out test sample for final evaluation.

The held-out test sample is not used during preprocessing-parameter estimation, hyperparameter selection, imbalance-option selection or predictor-domain contribution analysis.

In [7]:
# 7: Stratified training-test split

SPLIT_RANDOM_STATE = 314159
TEST_SIZE = 0.20

# Preserve the four-class outcome proportions across training and test samples.
training_index, test_index = train_test_split(
    np.arange(len(modelling_data)),
    test_size=TEST_SIZE,
    stratify=modelling_data["age18_outcome_code"],
    random_state=SPLIT_RANDOM_STATE,
)

training_data = modelling_data.iloc[training_index].copy()
test_data = modelling_data.iloc[test_index].copy()

assert len(training_data) == 7_619
assert len(test_data) == 1_905
assert set(training_data["NSID"]).isdisjoint(set(test_data["NSID"]))
assert len(training_data) + len(test_data) == len(modelling_data)

split_assignment = modelling_data[["NSID"]].copy()
split_assignment["sample"] = ""
split_assignment.loc[training_index, "sample"] = "Training"
split_assignment.loc[test_index, "sample"] = "Test"

assert split_assignment["sample"].isin(["Training", "Test"]).all()

# This file contains participant identifiers and remains in the local data_derived directory.
split_path = stage_4_directory / "stage_4_train_test_split.csv"
split_assignment.to_csv(split_path, index=False)

print(f"Training participants: {len(training_data):,}")
print(f"Test participants: {len(test_data):,}")
print(f"Saved: {split_path.relative_to(project_root)}")


Training participants: 7,619
Test participants: 1,905
Saved: data_derived\stage_4_modelling_design\stage_4_train_test_split.csv


In [8]:
# 8: Stratification check

# Verify that all four classes are represented in both samples.

split_balance = (
    modelling_data[["age18_outcome", "age18_outcome_code"]]
    .assign(sample=split_assignment["sample"].to_numpy())
    .groupby(["sample", "age18_outcome_code", "age18_outcome"])
    .size()
    .rename("Participants")
    .reset_index()
)

# transform("sum") returns the total for each sample on every row of that sample.
split_balance["Within-sample percentage"] = (
    split_balance["Participants"]
    / split_balance.groupby("sample")["Participants"].transform("sum")
    * 100
).round(2)

print("Class distribution by sample:")
display(split_balance)

assert split_balance.groupby("sample")["age18_outcome_code"].nunique().eq(4).all()


Class distribution by sample:


,sample,age18_outcome_code,age18_outcome,Participants,Within-sample percentage
0,Test,1,Education,991,52.02
1,Test,4,Employment,560,29.40
2,Test,5,Apprenticeship or training,104,5.46
3,Test,6,Unemployment or inactivity (NEET),250,13.12
4,Training,1,Education,3961,51.99
5,Training,4,Employment,2241,29.41
6,Training,5,Apprenticeship or training,416,5.46
7,Training,6,Unemployment or inactivity (NEET),1001,13.14


## Part 4: Nested cross-validation folds

The training sample uses five outer folds, with four inner folds constructed separately inside each outer-training sample.

- Inner folds select hyperparameters and the imbalance option.
- Outer validation folds estimate development performance.

Development performance is summarised across the five outer folds. Pooled outer-fold predictions are retained for diagnostic tables and confusion matrices.

In [9]:
# 9: Outer-fold assignments

OUTER_FOLDS = 5
OUTER_RANDOM_STATE = 271828

training_data = training_data.reset_index(drop=True)

# Stratification keeps the multiclass distribution represented in each outer fold.
outer_cv = StratifiedKFold(
    n_splits=OUTER_FOLDS,
    shuffle=True,
    random_state=OUTER_RANDOM_STATE,
)

outer_fold = np.zeros(len(training_data), dtype=int)
for fold_number, (_, validation_idx) in enumerate(
    outer_cv.split(training_data, training_data["age18_outcome_code"]),
    start=1,
):
    outer_fold[validation_idx] = fold_number

outer_assignments = pd.DataFrame({
    "NSID": training_data["NSID"],
    "outer_fold": outer_fold,
})

assert outer_assignments["outer_fold"].between(1, OUTER_FOLDS).all()
assert outer_assignments["NSID"].is_unique

# Fold-assignment files contain participant identifiers and remain in data_derived.
outer_path = stage_4_directory / "stage_4_outer_folds.csv"
outer_assignments.to_csv(outer_path, index=False)

print(outer_assignments["outer_fold"].value_counts().sort_index().to_string())
print(f"Saved: {outer_path.relative_to(project_root)}")


outer_fold
1    1524
2    1524
3    1524
4    1524
5    1523
Saved: data_derived\stage_4_modelling_design\stage_4_outer_folds.csv


In [10]:
# 10: Inner-fold assignments within each outer-training sample

INNER_FOLDS = 4
INNER_RANDOM_STATE_BASE = 161803

inner_records = []

# Recreate the inner split for each outer fold so outer-validation participants never enter inner CV.
for outer_fold_number in range(1, OUTER_FOLDS + 1):
    outer_training_mask = outer_assignments["outer_fold"].ne(outer_fold_number)
    outer_training = training_data.loc[outer_training_mask].reset_index(drop=True)

    # Inner folds are created only from the corresponding outer-training sample.
    inner_cv = StratifiedKFold(
        n_splits=INNER_FOLDS,
        shuffle=True,
        random_state=INNER_RANDOM_STATE_BASE + outer_fold_number,
    )

    inner_fold = np.zeros(len(outer_training), dtype=int)
    for inner_fold_number, (_, inner_validation_idx) in enumerate(
        inner_cv.split(outer_training, outer_training["age18_outcome_code"]),
        start=1,
    ):
        inner_fold[inner_validation_idx] = inner_fold_number

    inner_records.append(pd.DataFrame({
        "NSID": outer_training["NSID"],
        "outer_fold": outer_fold_number,
        "inner_fold": inner_fold,
    }))

inner_assignments = pd.concat(inner_records, ignore_index=True)

assert len(inner_assignments) == len(training_data) * (OUTER_FOLDS - 1)
assert inner_assignments["inner_fold"].between(1, INNER_FOLDS).all()
assert not inner_assignments.duplicated(["NSID", "outer_fold"]).any()

inner_path = stage_4_directory / "stage_4_inner_folds.csv"
inner_assignments.to_csv(inner_path, index=False)

print(f"Inner-fold rows: {len(inner_assignments):,}")
print(f"Saved: {inner_path.relative_to(project_root)}")


Inner-fold rows: 30,476
Saved: data_derived\stage_4_modelling_design\stage_4_inner_folds.csv


In [11]:
# 11: Fold class-coverage audit

# Confirm that every outer and inner validation fold contains all four classes.
fold_checks = []

for outer_fold_number in range(1, OUTER_FOLDS + 1):
    outer_validation_ids = set(
        outer_assignments.loc[
            outer_assignments["outer_fold"].eq(outer_fold_number), "NSID"
        ]
    )
    outer_validation = training_data[
        training_data["NSID"].isin(outer_validation_ids)
    ]

    fold_checks.append({
        "Level": "Outer validation",
        "Outer fold": outer_fold_number,
        "Inner fold": pd.NA,
        "Participants": len(outer_validation),
        "Classes": outer_validation["age18_outcome_code"].nunique(),
        "Smallest class": int(
            outer_validation["age18_outcome_code"].value_counts().min()
        ),
    })

    for inner_fold_number in range(1, INNER_FOLDS + 1):
        inner_validation_ids = set(
            inner_assignments.loc[
                inner_assignments["outer_fold"].eq(outer_fold_number)
                & inner_assignments["inner_fold"].eq(inner_fold_number),
                "NSID",
            ]
        )
        inner_validation = training_data[
            training_data["NSID"].isin(inner_validation_ids)
        ]

        fold_checks.append({
            "Level": "Inner validation",
            "Outer fold": outer_fold_number,
            "Inner fold": inner_fold_number,
            "Participants": len(inner_validation),
            "Classes": inner_validation["age18_outcome_code"].nunique(),
            "Smallest class": int(
                inner_validation["age18_outcome_code"].value_counts().min()
            ),
        })

fold_audit = pd.DataFrame(fold_checks)
assert fold_audit["Classes"].eq(4).all()
assert fold_audit["Smallest class"].gt(0).all()

fold_audit_path = stage_4_directory / "stage_4_fold_class_coverage_audit.csv"
fold_audit.to_csv(fold_audit_path, index=False)

fold_audit_summary = (
    fold_audit
    .groupby("Level", as_index=False)
    .agg(
        validation_folds=("Classes", "size"),
        minimum_participants=("Participants", "min"),
        maximum_participants=("Participants", "max"),
        minimum_smallest_class=("Smallest class", "min"),
        minimum_classes=("Classes", "min"),
    )
)

display(fold_audit_summary)
print(f"Saved: {fold_audit_path.relative_to(project_root)}")

,Level,validation_folds,minimum_participants,maximum_participants,minimum_smallest_class,minimum_classes
0,Inner validation,20,1523,1524,83,4
1,Outer validation,5,1523,1524,83,4


Saved: data_derived\stage_4_modelling_design\stage_4_fold_class_coverage_audit.csv


## Part 5: Model families and hyperparameter search

Core models are multinomial logistic regression (MLR), random forest (RF) and XGBoost. Balanced Random Forest (BRF) is supplementary.

MLR uses an exhaustive grid over nine `C` values. RF uses 60 fixed-seed random structural configurations, and BRF uses the same 60 tree structures. XGBoost uses 60 configurations, with five structures sampled within each of 12 learning-rate and boosting-round regimes.

The RF and XGBoost searches are non-exhaustive. Search spaces, candidate budgets and random states are fixed before model development. The same structural candidates are reused across outer folds and weighting modes.

In [12]:
# 12: Model-search specification

SEARCH_RANDOM_STATE = 424242
MODEL_RANDOM_STATE = 424242

RF_STRUCTURAL_CANDIDATES = 60
XGB_STRUCTURAL_CANDIDATES = 60
XGB_CONFIGURATIONS_PER_REGIME = 5

# Pair learning rate and boosting rounds so each prespecified regime receives the same search budget.
xgb_learning_round_pairs = [
    (0.03, 400),
    (0.03, 600),
    (0.03, 800),
    (0.05, 400),
    (0.05, 600),
    (0.05, 800),
    (0.10, 200),
    (0.10, 400),
    (0.10, 600),
    (0.20, 100),
    (0.20, 200),
    (0.20, 400),
]

assert len(xgb_learning_round_pairs) == 12
assert len(xgb_learning_round_pairs) * XGB_CONFIGURATIONS_PER_REGIME == XGB_STRUCTURAL_CANDIDATES

rf_search_parameters = {
    "n_estimators": [200, 400, 600, 800],
    "max_depth": [None, 8, 12, 16, 24, 32],
    "min_samples_leaf": [1, 2, 5, 10, 20],
    "max_features": ["sqrt", "log2", 0.3, 0.5, 0.8],
}

xgb_search_parameters = {
    "learning_rate_n_estimators_pairs": [list(pair) for pair in xgb_learning_round_pairs],
    "max_depth": [2, 3, 4, 5, 6, 8],
    "min_child_weight": [1, 3, 5, 10],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "reg_lambda": [0.5, 1.0, 5.0, 10.0],
    "gamma": [0.0, 0.1, 0.5, 1.0],
}

# Store parameter dictionaries as JSON strings so the design can be saved in a CSV register.
model_search_specification = pd.DataFrame([
    {
        "Model": "Multinomial logistic regression",
        "Role": "Core conventional baseline",
        "Search method": "Exhaustive grid",
        "Structural candidates": 9,
        "Search budget": "All 9 C values within each weighting mode",
        "Weighting modes": "None | Balanced",
        "Fixed parameters": json.dumps({
            "penalty": "l2",
            "solver": "lbfgs",
            "max_iter": 3000,
        }),
        "Search parameters": json.dumps({
            "C": [0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100],
        }),
    },
    {
        "Model": "Random forest",
        "Role": "Core bagging-based ML model",
        "Search method": "Fixed-seed random search without replacement",
        "Structural candidates": RF_STRUCTURAL_CANDIDATES,
        "Search budget": "60 structural configurations per weighting mode",
        "Weighting modes": "None | Balanced",
        "Fixed parameters": json.dumps({
            "min_samples_split": 2,
            "criterion": "gini",
            "n_jobs": -1,
            "random_state": MODEL_RANDOM_STATE,
        }),
        "Search parameters": json.dumps(rf_search_parameters),
    },
    {
        "Model": "XGBoost",
        "Role": "Core boosting-based ML model",
        "Search method": "Regime-balanced fixed-seed random search",
        "Structural candidates": XGB_STRUCTURAL_CANDIDATES,
        "Search budget": "60 structural configurations: 5 within each of 12 learning-rate/round regimes",
        "Weighting modes": "None | Balanced sample weights",
        "Fixed parameters": json.dumps({
            "objective": "multi:softprob",
            "num_class": 4,
            "eval_metric": "mlogloss",
            "tree_method": "hist",
            "n_jobs": -1,
            "random_state": MODEL_RANDOM_STATE,
        }),
        "Search parameters": json.dumps(xgb_search_parameters),
    },
    {
        "Model": "Balanced random forest",
        "Role": "Supplementary imbalance-aware ensemble",
        "Search method": "Same 60 structural configurations as random forest",
        "Structural candidates": RF_STRUCTURAL_CANDIDATES,
        "Search budget": "60 structural configurations",
        "Weighting modes": "Internal balanced sampling only",
        "Fixed parameters": json.dumps({
            "min_samples_split": 2,
            "criterion": "gini",
            "sampling_strategy": "all",
            "replacement": True,
            "bootstrap": False,
            "class_weight": None,
            "n_jobs": -1,
            "random_state": MODEL_RANDOM_STATE,
        }),
        "Search parameters": json.dumps(rf_search_parameters),
    },
])

model_search_specification["Selection metric"] = "Macro F1"
model_search_specification["Search random state"] = SEARCH_RANDOM_STATE

model_spec_path = stage_4_directory / "stage_4_model_search_specification.csv"
model_search_specification.to_csv(model_spec_path, index=False)

display(model_search_specification[[
    "Model",
    "Role",
    "Search method",
    "Structural candidates",
    "Search budget",
    "Weighting modes",
    "Selection metric",
]])
print(f"Saved: {model_spec_path.relative_to(project_root)}")


,Model,Role,Search method,Structural candidates,Search budget,Weighting modes,Selection metric
0,Multinomial logistic regression,Core conventional baseline,Exhaustive grid,9,All 9 C values within each weighting mode,None | Balanced,Macro F1
1,Random forest,Core bagging-based ML model,Fixed-seed random search without replacement,60,60 structural configurations per weighting mode,None | Balanced,Macro F1
2,XGBoost,Core boosting-based ML model,Regime-balanced fixed-seed random search,60,60 structural configurations: 5 within each of...,None | Balanced sample weights,Macro F1
3,Balanced random forest,Supplementary imbalance-aware ensemble,Same 60 structural configurations as random fo...,60,60 structural configurations,Internal balanced sampling only,Macro F1


Saved: data_derived\stage_4_modelling_design\stage_4_model_search_specification.csv


### Imbalance handling

MLR and RF compare unweighted fitting with `class_weight="balanced"`. XGBoost compares unweighted fitting with balanced sample weights calculated from each training fit. BRF uses internal balanced sampling without additional class weighting.

No resampling is applied before data partitioning or cross-validation.

## Part 6: Evaluation specification

**Selection metric**
- Macro F1

**Overall evaluation**
- Balanced accuracy
- Matthews correlation coefficient (MCC)
- Accuracy
- Weighted F1

**Class-specific evaluation**
- Precision
- Recall
- F1
- Support

**Diagnostics**
- Confusion-matrix counts
- Row-normalised confusion matrix
- Observed and predicted class distributions
- Majority-class dummy baseline

Only macro F1 is used for hyperparameter and imbalance-option selection.


In [13]:
# 13: Evaluation register

# Store metric roles so model development uses one selection criterion consistently.

evaluation_register = pd.DataFrame([
    {"Measure": "Macro F1", "Role": "Primary and tuning criterion", "Level": "Overall"},
    {"Measure": "Balanced accuracy", "Role": "Key secondary", "Level": "Overall"},
    {"Measure": "MCC", "Role": "Key secondary", "Level": "Overall"},
    {"Measure": "Accuracy", "Role": "Supporting", "Level": "Overall"},
    {"Measure": "Weighted F1", "Role": "Supporting", "Level": "Overall"},
    {"Measure": "Precision", "Role": "Required", "Level": "Class-specific"},
    {"Measure": "Recall", "Role": "Required", "Level": "Class-specific"},
    {"Measure": "F1", "Role": "Required", "Level": "Class-specific"},
    {"Measure": "Support", "Role": "Required", "Level": "Class-specific"},
])

evaluation_register_path = stage_4_directory / "stage_4_evaluation_register.csv"
evaluation_register.to_csv(evaluation_register_path, index=False)

display(evaluation_register)
print(f"Saved: {evaluation_register_path.relative_to(project_root)}")


,Measure,Role,Level
0,Macro F1,Primary and tuning criterion,Overall
1,Balanced accuracy,Key secondary,Overall
2,MCC,Key secondary,Overall
3,Accuracy,Supporting,Overall
4,Weighted F1,Supporting,Overall
5,Precision,Required,Class-specific
6,Recall,Required,Class-specific
7,F1,Required,Class-specific
8,Support,Required,Class-specific


Saved: data_derived\stage_4_modelling_design\stage_4_evaluation_register.csv


## Part 7: Predictor-domain contribution design

Two domain-level analyses use the training sample and the same outer folds as model development.

**Leave-one-domain-out omission** compares the full 69-predictor specification with each of the 10 represented domains omitted in turn. The model specification selected within each outer fold is held fixed, and performance is compared on the same outer-validation participants.

**Common-baseline separate addition** uses demographic background, family socioeconomic background, SEN/disability/health, and school/local context as a 23-predictor reference set. Each of the six remaining domains is added separately to this same baseline.

MLR, RF, XGBoost and supplementary BRF are evaluated. Macro F1 is the primary performance change; balanced accuracy and class-specific recall and F1 are supporting measures.

In [14]:
# 14: Stage 5.1 domain-contribution specification

# Register the 10 represented domains and the two domain-level analyses.

# Group the manifest so every represented domain carries its retained predictor list.
domain_register = (
    predictor_manifest[["Predictor domain", "Predictor"]]
    .groupby("Predictor domain")["Predictor"]
    .agg(list)
    .reset_index()
)
domain_register["Predictors"] = domain_register["Predictor"].apply(len)

domain_omission_register = domain_register.loc[
    domain_register["Predictors"].gt(0)
].copy()

stage_5_1_models = (
    "MLR | Random forest | XGBoost | Balanced random forest"
)

domain_omission_register["Analysis"] = "Leave-one-domain-out"
domain_omission_register["Models"] = stage_5_1_models
domain_omission_register["Reference"] = "Full 69-predictor set"
domain_omission_register["Primary change"] = "Full minus omitted macro F1"
domain_omission_register["Supporting changes"] = (
    "Full minus omitted balanced accuracy | "
    "class-specific recall | class-specific F1"
)

represented_domains = domain_omission_register["Predictor domain"].tolist()
represented_domain_set = set(represented_domains)
predictor_count_by_domain = (
    domain_omission_register
    .set_index("Predictor domain")["Predictors"]
    .to_dict()
)

common_baseline_domains = [
    "Demographic background",
    "Family socioeconomic background",
    "SEN, disability and health",
    "School and local context",
]

separate_addition_domains = [
    "Educational aspirations and post-16 plans",
    "School experiences and engagement",
    "Psychosocial characteristics",
    "Experiences and behaviours",
    "Parental attitudes, support and engagement",
    "Post-16 social influences and guidance",
]

# The baseline and addition groups must partition the 10 represented domains without overlap.
assert set(common_baseline_domains).isdisjoint(separate_addition_domains)
assert set(common_baseline_domains + separate_addition_domains) == represented_domain_set

baseline_predictor_count = sum(
    predictor_count_by_domain[domain]
    for domain in common_baseline_domains
)

addition_rows = [
    {
        "Analysis": "Common-baseline reference",
        "Specification": "Common baseline",
        "Added domain": pd.NA,
        "Included domains": " | ".join(common_baseline_domains),
        "Baseline predictors": baseline_predictor_count,
        "Added predictors": 0,
        "Predictors": baseline_predictor_count,
        "Reference": "Not applicable",
        "Models": stage_5_1_models,
        "Primary change": "Reference performance",
    }
]

for domain in separate_addition_domains:
    added_predictors = int(predictor_count_by_domain[domain])
    addition_rows.append(
        {
            "Analysis": "Separate domain addition",
            "Specification": f"Common baseline + {domain}",
            "Added domain": domain,
            "Included domains": " | ".join(
                common_baseline_domains + [domain]
            ),
            "Baseline predictors": baseline_predictor_count,
            "Added predictors": added_predictors,
            "Predictors": baseline_predictor_count + added_predictors,
            "Reference": "Common baseline",
            "Models": stage_5_1_models,
            "Primary change": "Augmented minus baseline macro F1",
        }
    )

domain_addition_register = pd.DataFrame(addition_rows)

assert len(domain_omission_register) == 10
assert domain_omission_register["Predictors"].sum() == 69
assert len(domain_addition_register) == 7
assert domain_addition_register["Analysis"].eq(
    "Common-baseline reference"
).sum() == 1
assert domain_addition_register["Analysis"].eq(
    "Separate domain addition"
).sum() == 6
assert set(
    domain_addition_register.loc[
        domain_addition_register["Analysis"].eq("Separate domain addition"),
        "Added domain",
    ]
) == set(separate_addition_domains)

omission_path = (
    stage_4_directory
    / "stage_4_domain_omission_specification.csv"
)
addition_path = (
    stage_4_directory
    / "stage_4_domain_addition_specification.csv"
)

domain_omission_register.to_csv(omission_path, index=False)
domain_addition_register.to_csv(addition_path, index=False)

print("Primary domain analysis:")
display(
    domain_omission_register[
        ["Predictor domain", "Predictors", "Analysis", "Models"]
    ]
)

print("\nSecondary domain analysis:")
display(
    domain_addition_register[
        [
            "Specification",
            "Added domain",
            "Baseline predictors",
            "Added predictors",
            "Predictors",
            "Models",
        ]
    ]
)

print(f"\nSaved: {omission_path.relative_to(project_root)}")
print(f"Saved: {addition_path.relative_to(project_root)}")


Primary domain analysis:


,Predictor domain,Predictors,Analysis,Models
0,Demographic background,5,Leave-one-domain-out,MLR | Random forest | XGBoost | Balanced rando...
1,Educational aspirations and post-16 plans,2,Leave-one-domain-out,MLR | Random forest | XGBoost | Balanced rando...
2,Experiences and behaviours,7,Leave-one-domain-out,MLR | Random forest | XGBoost | Balanced rando...
3,Family socioeconomic background,7,Leave-one-domain-out,MLR | Random forest | XGBoost | Balanced rando...
4,"Parental attitudes, support and engagement",16,Leave-one-domain-out,MLR | Random forest | XGBoost | Balanced rando...
5,Post-16 social influences and guidance,10,Leave-one-domain-out,MLR | Random forest | XGBoost | Balanced rando...
6,Psychosocial characteristics,2,Leave-one-domain-out,MLR | Random forest | XGBoost | Balanced rando...
7,"SEN, disability and health",8,Leave-one-domain-out,MLR | Random forest | XGBoost | Balanced rando...
8,School and local context,3,Leave-one-domain-out,MLR | Random forest | XGBoost | Balanced rando...
9,School experiences and engagement,9,Leave-one-domain-out,MLR | Random forest | XGBoost | Balanced rando...



Secondary domain analysis:


,Specification,Added domain,Baseline predictors,Added predictors,Predictors,Models
0,Common baseline,<NA>,23,0,23,MLR | Random forest | XGBoost | Balanced rando...
1,Common baseline + Educational aspirations and ...,Educational aspirations and post-16 plans,23,2,25,MLR | Random forest | XGBoost | Balanced rando...
2,Common baseline + School experiences and engag...,School experiences and engagement,23,9,32,MLR | Random forest | XGBoost | Balanced rando...
3,Common baseline + Psychosocial characteristics,Psychosocial characteristics,23,2,25,MLR | Random forest | XGBoost | Balanced rando...
4,Common baseline + Experiences and behaviours,Experiences and behaviours,23,7,30,MLR | Random forest | XGBoost | Balanced rando...
5,"Common baseline + Parental attitudes, support ...","Parental attitudes, support and engagement",23,16,39,MLR | Random forest | XGBoost | Balanced rando...
6,Common baseline + Post-16 social influences an...,Post-16 social influences and guidance,23,10,33,MLR | Random forest | XGBoost | Balanced rando...



Saved: data_derived\stage_4_modelling_design\stage_4_domain_omission_specification.csv
Saved: data_derived\stage_4_modelling_design\stage_4_domain_addition_specification.csv


## Part 8: Final held-out evaluation rule

For Stage 6, each model family is tuned on the complete training sample using the same fixed search design and macro-F1 criterion. The selected configuration is fitted to the complete training sample and evaluated once on the held-out test sample.

Stage 6 reports the specified overall and class-specific metrics. Uncertainty is assessed with 5,000 paired stratified bootstrap resamples of the held-out predictions. The same resample indices are used for all models.


## Part 9: Weighting and evaluation

Class weighting addresses unequal outcome frequencies in model fitting; it is not a survey or attrition weight.

Performance is evaluated within the modelling sample, with the held-out test used for internal validation.

In [15]:
# 15: Stage 4 design audit

# Confirm that all fixed design components are internally consistent.
design_checks = pd.DataFrame([
    {"Check": "Final modelling sample has 9,524 participants", "Passed": len(modelling_data) == 9_524},
    {"Check": "All 69 predictors are represented once", "Passed": len(predictor_roles) == 69 and predictor_roles["Predictor"].is_unique},
    {"Check": "All four outcome classes are present", "Passed": modelling_data["age18_outcome_code"].nunique() == 4},
    {"Check": "Training and test identifiers do not overlap", "Passed": set(training_data["NSID"]).isdisjoint(set(test_data["NSID"]))},
    {"Check": "Training sample contains 7,619 participants", "Passed": len(training_data) == 7_619},
    {"Check": "Test sample contains 1,905 participants", "Passed": len(test_data) == 1_905},
    {"Check": "Five outer folds are defined", "Passed": outer_assignments["outer_fold"].nunique() == 5},
    {"Check": "Four inner folds are defined within every outer-training sample", "Passed": inner_assignments.groupby("outer_fold")["inner_fold"].nunique().eq(4).all()},
    {"Check": "Every validation fold contains all four classes", "Passed": fold_audit["Classes"].eq(4).all()},
    {"Check": "Fold class-coverage audit is saved", "Passed": fold_audit_path.is_file()},
    {"Check": "Evaluation register is saved", "Passed": evaluation_register_path.is_file()},
    {"Check": "Macro F1 is the single selection metric", "Passed": model_search_specification["Selection metric"].eq("Macro F1").all()},
    {"Check": "Core model families are MLR, RF and XGBoost", "Passed": set(model_search_specification.loc[model_search_specification["Role"].str.startswith("Core"), "Model"]) == {"Multinomial logistic regression", "Random forest", "XGBoost"}},
    {"Check": "Balanced random forest is supplementary", "Passed": model_search_specification.loc[model_search_specification["Model"].eq("Balanced random forest"), "Role"].eq("Supplementary imbalance-aware ensemble").all()},
    {"Check": "RF uses 60 structural candidates", "Passed": int(model_search_specification.loc[model_search_specification["Model"].eq("Random forest"), "Structural candidates"].iloc[0]) == 60},
    {"Check": "XGBoost uses 60 structural candidates", "Passed": int(model_search_specification.loc[model_search_specification["Model"].eq("XGBoost"), "Structural candidates"].iloc[0]) == 60},
    {"Check": "XGBoost allocates five configurations to each of 12 regimes", "Passed": len(xgb_learning_round_pairs) == 12 and XGB_CONFIGURATIONS_PER_REGIME == 5},
    {"Check": "BRF uses the RF structural search space", "Passed": model_search_specification.loc[model_search_specification["Model"].eq("Balanced random forest"), "Search parameters"].iloc[0] == model_search_specification.loc[model_search_specification["Model"].eq("Random forest"), "Search parameters"].iloc[0]},
    {"Check": "Ten non-empty domains are specified for Stage 5.1 omission", "Passed": len(domain_omission_register) == 10},
    {"Check": "One common baseline and six separate additions are specified", "Passed": len(domain_addition_register) == 7 and domain_addition_register["Analysis"].eq("Common-baseline reference").sum() == 1 and domain_addition_register["Analysis"].eq("Separate domain addition").sum() == 6},
    {"Check": "Baseline and addition domains partition the ten represented domains", "Passed": set(common_baseline_domains).isdisjoint(separate_addition_domains) and set(common_baseline_domains + separate_addition_domains) == represented_domain_set},
    {"Check": "Stage 5.1 includes MLR, RF, XGBoost and BRF", "Passed": domain_omission_register["Models"].eq(stage_5_1_models).all() and domain_addition_register["Models"].eq(stage_5_1_models).all()},
])

assert design_checks["Passed"].all()

audit_path = stage_4_directory / "stage_4_design_audit.csv"
design_checks.to_csv(audit_path, index=False)

display(design_checks)
print(f"Saved: {audit_path.relative_to(project_root)}")


,Check,Passed
0,"Final modelling sample has 9,524 participants",True
1,All 69 predictors are represented once,True
2,All four outcome classes are present,True
3,Training and test identifiers do not overlap,True
4,"Training sample contains 7,619 participants",True
5,"Test sample contains 1,905 participants",True
6,Five outer folds are defined,True
7,Four inner folds are defined within every oute...,True
8,Every validation fold contains all four classes,True
9,Fold class-coverage audit is saved,True


Saved: data_derived\stage_4_modelling_design\stage_4_design_audit.csv


## Stage 4 summary

Stage 4 defines the 80:20 stratified train-test partition, five outer folds with four inner folds, preprocessing roles, model-search specifications, evaluation measures and two predictor-domain contribution designs.

Macro F1 is the single model-selection criterion. The held-out test sample is reserved from model development and predictor-domain analysis until final evaluation.